In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'python').exists():
    ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Working directory: {ROOT}')


In [1]:
"""
RNN classification using Transformer-encoded word embeddings.
Encodes each word using a transformer encoder (intfloat/multilingual-e5-base),
then passes word embeddings to an RNN.


TOKENIZE ENCODE WORDS, PASS THEM TO RNN, EMBEDDINGS SIZE 768
"""

import os
import gc
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score
)
from datetime import datetime
from transformers import AutoTokenizer, AutoModel
import string

from python.preprocessing_class import Preprocessing, load_pres, load_movies

# ─────────────────────────────────-────────
# CONFIGURATION !!!!
# ─────────────────────────────────────────

# choose dataset
DATASET = "movie"
MAX_LENGHT_TOKEN = 1024
SAVING_FILE_NAME =  DATASET + "_rnn_encoder"

# -----------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20

BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 1e-5
EARLY_STOPPING_PATIENCE = 10

RNN_TYPE = 'gru'  # 'lstm' or 'gru'
HIDDEN_DIM = 32
NUM_LAYERS = 2
DROPOUT = 0.4

# Transformer encoder config
TRANSFORMER_MODEL = "intfloat/multilingual-e5-base"  # 768-dim embeddings
FREEZE_TRANSFORMER = True  # Don't train transformer, only RNN

# ─────────────────────────────────────────
# DEVICE
# ─────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU memory available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")


/home/alant/repos/tal-projet/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU memory available: 6.03 GB


In [2]:

# ─────────────────────────────────────────
# LOAD TRANSFORMER ENCODER
# ─────────────────────────────────────────

print(f"\nLoading transformer encoder ({TRANSFORMER_MODEL})...")
tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL)
encoder = AutoModel.from_pretrained(TRANSFORMER_MODEL)  # Keep on CPU for encoding phase

# Freeze transformer weights (don't train it)
if FREEZE_TRANSFORMER:
    for param in encoder.parameters():
        param.requires_grad = False
    print("Transformer encoder frozen (weights not trainable)")

EMBED_DIM = encoder.config.hidden_size  # Usually 768
print(f"Transformer embedding dimension: {EMBED_DIM}")


# ─────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────
print("\nLoading data...")

if DATASET == "pres":
    alltxt, alllabs = load_pres("Dataset/corpus.tache1.learn.utf8")
    alltxt = np.array(alltxt)
    # Convert labels: 1 -> 0, -1 -> 1
    alllabs = np.where(np.array(alllabs) == 1, 0, 1)
    print(f"  Total samples: {len(alltxt)}")
    print(f"  Label distribution: {np.bincount(alllabs)}")
else:
    path = "./Dataset/movies1000/"
    alltxt,alllabs = load_movies(path)
    alltxt, alllabs = np.array(alltxt), np.array(alllabs)
    print(f"  Total samples: {len(alltxt)}")
    print(f"  Label distribution: {np.bincount(alllabs)}")

# Train/val/test split
X_train, X_test, y_train, y_test = train_test_split(
    alltxt, alllabs,
    test_size=TEST_SIZE,
    stratify=alllabs,
    random_state=RANDOM_STATE
)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train,
    test_size=VAL_SIZE,
    stratify=y_train,
    random_state=RANDOM_STATE
)

print(f"  Train: {len(X_train_sub)}, Val: {len(X_val)}, Test: {len(X_test)}")

# ─────────────────────────────────────────
# PREPROCESSING
# ─────────────────────────────────────────

print("\nSetting up preprocessing...")

punc = set(string.punctuation + '\n\r\t')
punc.discard("'")
custom_punctuation = "".join(punc)


lang = "french" if DATASET == "pres" else "english"

prep = Preprocessing(
        low_case=True,
        rm_punctuation=True,
        rm_number=False,
        word_norm=None,
        pos_tagging=False,
        all_capital=True,
        cap_name=True,
        rm_accent=False,
        lang=lang,
        punct=custom_punctuation,
        urls=False
)

print("Preprocessing language ", lang)

print("Preprocessing texts...")
X_train_prep = [prep.process(t) for t in X_train_sub]
X_val_prep = [prep.process(t) for t in X_val]
X_test_prep = [prep.process(t) for t in X_test]

# ─────────────────────────────────────────
# ENCODE WORDS WITH TRANSFORMER
# ─────────────────────────────────────────

def encode_texts_to_word_embeddings(texts, tokenizer, encoder, device, max_length=MAX_LENGHT_TOKEN):
    """
    Encode each text's words using transformer encoder.
    Returns list of sequences where each element is a word embedding (EMBED_DIM vector).
    Skips texts that have no actual word tokens after removing special tokens.
    """
    embeddings_list = []
    valid_indices = []  # Track which original indices are kept
    
    print(f"Encoding {len(texts)} texts to word embeddings...")
    for idx, text in enumerate(texts):
        if (idx + 1) % 500 == 0:
            print(f"  Encoded {idx + 1}/{len(texts)}")
        
        # Tokenize text into words
        # Add special tokens, truncate, and pad
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            return_tensors="pt",
            padding=False
        )
        
        # Keep on CPU for encoding (encoder is on CPU, embeddings will be moved to GPU during training)
        input_ids = encoded['input_ids']
        attention_mask = encoded['attention_mask']
        
        # Get token embeddings from transformer (on CPU)
        with torch.no_grad():
            outputs = encoder(input_ids=input_ids, attention_mask=attention_mask)
            # outputs.last_hidden_state has shape (1, num_tokens, EMBED_DIM)
            token_embeddings = outputs.last_hidden_state[0]  # (num_tokens, EMBED_DIM)
        
        # Remove [CLS] and [SEP] tokens (first and last)
        # Keep only actual word tokens
        word_embeddings = token_embeddings[1:-1]  # Exclude special tokens
        
        # Skip texts with no actual word tokens (too short)
        if len(word_embeddings) > 0:
            embeddings_list.append(word_embeddings.detach().cpu())
            valid_indices.append(idx)
    
    return embeddings_list, valid_indices

print("Encoding train texts...")
X_train_embeddings, train_valid_idx = encode_texts_to_word_embeddings(X_train_prep, tokenizer, encoder, device)
y_train_sub = y_train_sub[train_valid_idx]
print(f"  Kept {len(X_train_embeddings)}/{len(X_train_prep)} train samples")

print("Encoding val texts...")
X_val_embeddings, val_valid_idx = encode_texts_to_word_embeddings(X_val_prep, tokenizer, encoder, device)
y_val = y_val[val_valid_idx]
print(f"  Kept {len(X_val_embeddings)}/{len(X_val_prep)} val samples")

print("Encoding test texts...")
X_test_embeddings, test_valid_idx = encode_texts_to_word_embeddings(X_test_prep, tokenizer, encoder, device)
y_test = y_test[test_valid_idx]
print(f"  Kept {len(X_test_embeddings)}/{len(X_test_prep)} test samples")

# ─────────────────────────────────────────
# DATASET & DATALOADER
# ─────────────────────────────────────────

def collate_fn(batch):
    """Collate function to pad embedding sequences in a batch."""
    embeddings, labels = zip(*batch)
    
    # Get lengths before padding
    lengths = torch.tensor([len(emb) for emb in embeddings], dtype=torch.long)
    
    # Pad sequences to same length
    padded = pad_sequence(embeddings, batch_first=True, padding_value=0.0)
    labels = torch.tensor(labels, dtype=torch.long)
    
    return padded, lengths, labels


class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


dataset_train = EmbeddingDataset(X_train_embeddings, y_train_sub)
dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

dataset_val = EmbeddingDataset(X_val_embeddings, y_val)
dataloader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

dataset_test = EmbeddingDataset(X_test_embeddings, y_test)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print("Datasets ready")

# ─────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────

class RNNClassifier(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_layers=1, 
                 dropout=0.3, rnn_type='lstm'):
        super().__init__()
        self.rnn_type = rnn_type.lower()
        
        # Note: embed_dim is already provided by transformer (no embedding layer needed)
        rnn_class = nn.LSTM if self.rnn_type == 'lstm' else nn.GRU
        
        self.rnn = rnn_class(
            input_size=embed_dim,  # Input is pre-embedded (768-dim from transformer)
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, 2)

    def forward(self, x, lengths):
        # x has shape (batch_size, seq_len, embed_dim)
        # No embedding needed since input is already embedded by transformer
        
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        
        if self.rnn_type == 'lstm':
            _, (hn, _) = self.rnn(packed)
        else:  # GRU
            _, hn = self.rnn(packed)
        
        hn = torch.cat([hn[-2], hn[-1]], dim=1)  # concat forward + backward
        return self.classifier(self.dropout(hn))

print(f"\nLoading model ({RNN_TYPE})...")
model = RNNClassifier(
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    rnn_type=RNN_TYPE,
).to(device)

print(model)

# ─────────────────────────────────────────
# LOSS & OPTIMIZER
# ─────────────────────────────────────────

def compute_class_weights(labels):
    counts = torch.bincount(torch.tensor(labels)).float()
    weights = 1.0 / counts
    weights = weights / weights.sum()
    return weights

class_weights = compute_class_weights(y_train_sub).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

# ─────────────────────────────────────────
# METRICS HELPER
# ─────────────────────────────────────────

def compute_metrics(all_labels, all_preds, all_probs):
    """Compute evaluation metrics."""
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    
    if DATASET == "pres":
        return {
            "f1": float(f1_score(all_labels, all_preds, average="binary", zero_division=0)),
            "precision": float(precision_score(all_labels, all_preds, zero_division=0)),
            "recall": float(recall_score(all_labels, all_preds, zero_division=0)),
            "roc_auc": float(roc_auc_score(all_labels, all_probs)),
            "avg_precision": float(average_precision_score(all_labels, all_probs)),
        }
    
    return {
        "f1": float(f1_score(all_labels, all_preds, average="binary", zero_division=0)),
        "precision": float(precision_score(all_labels, all_preds, zero_division=0)),
        "recall": float(recall_score(all_labels, all_preds, zero_division=0)),
        "roc_auc": float(roc_auc_score(all_labels, all_probs)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
    }


# ─────────────────────────────────────────
# TRAIN / EVAL LOOP
# ─────────────────────────────────────────

def run_epoch(model, loader, optimizer, criterion, training=True):
    """Run one epoch of training or evaluation."""
    model.train() if training else model.eval()

    total_loss = 0
    all_labels, all_preds, all_probs = [], [], []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for embeddings, lengths, labels in loader:
            embeddings = embeddings.to(device)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad()

            logits = model(embeddings, lengths)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            total_loss += loss.item()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].detach().cpu().numpy())

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = float(total_loss / len(loader))
    return metrics



Loading transformer encoder (intfloat/multilingual-e5-base)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6423.18it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Transformer encoder frozen (weights not trainable)
Transformer embedding dimension: 768

Loading data...
  Total samples: 2000
  Label distribution: [1000 1000]
  Train: 1280, Val: 320, Test: 400

Setting up preprocessing...
Preprocessing language  english
Preprocessing texts...
Encoding train texts...
Encoding 1280 texts to word embeddings...


RuntimeError: index 514 is out of bounds for dimension 1 with size 514

In [ ]:

# ─────────────────────────────────────────
# TRAINING WITH EARLY STOPPING
# ─────────────────────────────────────────

print("\n" + "="*80)
print("TRAINING")
print("="*80)

history = []
best_f1 = 0.0
best_model_state = None
patience_counter = 0

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}", end=" ")

    train_m = run_epoch(model, dataloader_train, optimizer, criterion, training=True)
    val_m = run_epoch(model, dataloader_val, optimizer, criterion, training=False)

    # Early stopping
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        improvement_marker = "↑ (BEST)"
    else:
        patience_counter += 1
        improvement_marker = f"(patience {patience_counter}/{EARLY_STOPPING_PATIENCE})"

    history.append({
        "epoch": epoch + 1,
        "train": train_m,
        "val": val_m,
    })

    print(
        f"{improvement_marker:25s} | "
        f"loss  train {train_m['loss']:.4f}  val {val_m['loss']:.4f} | "
        f"f1    train {train_m['f1']:.3f}  val {val_m['f1']:.3f} | "
        f"auc   train {train_m['roc_auc']:.3f}  val {val_m['roc_auc']:.3f} | "
        f"prec  train {train_m['precision']:.3f}  val {val_m['precision']:.3f} | "
        f"rec   train {train_m['recall']:.3f}  val {val_m['recall']:.3f}"
    )

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}: No improvement for {EARLY_STOPPING_PATIENCE} epochs")
        break

print(f"\nBest val F1: {best_f1:.3f}")

# ─────────────────────────────────────────
# TEST EVALUATION
# ─────────────────────────────────────────

print("\n" + "="*80)
print("TEST EVALUATION")
print("="*80)

model.load_state_dict(best_model_state)
torch.save(model.state_dict(), f"{SAVING_FILE_NAME}.pth")

test_m = run_epoch(model, dataloader_test, optimizer, criterion, training=False)

print(
    f"\nTest | loss {test_m['loss']:.4f} | f1 {test_m['f1']:.4f} | "
    f"prec {test_m['precision']:.4f} | rec {test_m['recall']:.4f} | "
    f"auc {test_m['roc_auc']:.4f} | ap {test_m['avg_precision']:.4f}"
)

# ─────────────────────────────────────────
# SAVE RESULTS
# ─────────────────────────────────────────

results = {
    "run_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "device": str(device),
    "transformer_encoder": {
        "model": TRANSFORMER_MODEL,
        "embed_dim": EMBED_DIM,
        "frozen": FREEZE_TRANSFORMER,
    },
    "model_config": {
        "rnn_type": RNN_TYPE,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 1e-5,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    },
    "best_val_f1": best_f1,
    "epochs_trained": len(history),
    "history": history,
    "test": test_m,
}

with open(f"{SAVING_FILE_NAME}.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to {SAVING_FILE_NAME}.json")
print(f"Model saved to {SAVING_FILE_NAME}.pth")

print("DATASET ", DATASET)
print(f"\nLoaded transformer encoder ({TRANSFORMER_MODEL})...")
print(f"Transformer embedding dimension: {EMBED_DIM}")
print(f"\nLoaded model ({RNN_TYPE})...")

In [ ]:
"""
RNN sentiment classification using CountVectorizer
Tests a single configuration with clean evaluation structure
"""

# 2 is corrected function

import os
import gc
import json
import codecs
import re
import numpy as np
import string
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    f1_score, average_precision_score, roc_auc_score, 
    precision_score, recall_score, accuracy_score
)
from datetime import datetime

from python.preprocessing_class import Preprocessing, load_movies, load_pres
from nltk.corpus import stopwords

final_stopwords_list = stopwords.words('french')

# ─────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────
VECTORIZER_TYPE = "count"  # 'count' or 'tfidf'
RNN_TYPE = 'gru'  # 'lstm' or 'gru'
DATASET = "movie"
SAVING_FILE_NAME =  DATASET + "_rnn_" + RNN_TYPE + " " + VECTORIZER_TYPE

RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20

BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 1e-5
EARLY_STOPPING_PATIENCE = 10

EMBED_DIM = 64
HIDDEN_DIM = 32
NUM_LAYERS = 2
DROPOUT = 0.4

# Vectorizer config
MAX_FEATURES = 10000 # changed to 10000!
NGRAM_RANGE = (1, 1)  # (1, 1) unigram, (1, 2) bigram

# ─────────────────────────────────────────
# DEVICE
# ─────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU memory available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")

# ─────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────

print("Loading data...")
if DATASET == "pres":
    alltxt, alllabs = load_pres("Dataset/corpus.tache1.learn.utf8")
    alltxt = np.array(alltxt)
    # Convert labels: 1 -> 0, -1 -> 1
    alllabs = np.where(np.array(alllabs) == 1, 0, 1)
    print(f"  Total samples: {len(alltxt)}")
    print(f"  Label distribution: {np.bincount(alllabs)}")
else:
    path = "./Dataset/movies1000/"
    alltxt,alllabs = load_movies(path)
    alltxt, alllabs = np.array(alltxt), np.array(alllabs)
    print(f"  Total samples: {len(alltxt)}")
    print(f"  Label distribution: {np.bincount(alllabs)}")


# Train/val/test split
X_train, X_test, y_train, y_test = train_test_split(
    alltxt, alllabs,
    test_size=TEST_SIZE,
    stratify=alllabs,
    random_state=RANDOM_STATE
)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train,
    test_size=VAL_SIZE,
    stratify=y_train,
    random_state=RANDOM_STATE
)

print(f"  Train: {len(X_train_sub)}, Val: {len(X_val)}, Test: {len(X_test)}")

# ─────────────────────────────────────────
# PREPROCESSING
# ─────────────────────────────────────────

print("\nSetting up preprocessing...")

punc = set(string.punctuation + '\n\r\t')
punc.discard("'")
custom_punctuation = "".join(punc)

lang = "french" if DATASET == "pres" else "english"

prep = Preprocessing(
    low_case=True,
    rm_punctuation=True,
    rm_number=False,
    word_norm=None,
    pos_tagging=False,
    all_capital=True,
    cap_name=True,
    rm_accent=False,
    lang=lang,
    punct=custom_punctuation,
    urls=False
)

print("Preprocessing train...")
X_train_prep = [prep.process(t) for t in X_train_sub]
print("Preprocessing val...")
X_val_prep = [prep.process(t) for t in X_val]
print("Preprocessing test...")
X_test_prep = [prep.process(t) for t in X_test]

# ─────────────────────────────────────────
# VECTORIZATION
# ─────────────────────────────────────────

print(f"\nVectorizing with {VECTORIZER_TYPE}...")

if VECTORIZER_TYPE == "tfidf":
    vectorizer = TfidfVectorizer(
        max_features=MAX_FEATURES,
        max_df=0.95,
        min_df=2,
        ngram_range=NGRAM_RANGE,
        stop_words=final_stopwords_list
    )
else:
    vectorizer = CountVectorizer(
        max_features=MAX_FEATURES,
        max_df=0.95,
        min_df=2,
        ngram_range=NGRAM_RANGE,
        stop_words=final_stopwords_list
    )

X_train_vec = vectorizer.fit_transform(X_train_prep)
X_val_vec = vectorizer.transform(X_val_prep)
X_test_vec = vectorizer.transform(X_test_prep)

print(f"  Vocabulary size: {X_train_vec.shape[1]}")

# ─────────────────────────────────────────
# CONVERT TO SEQUENCES
# ─────────────────────────────────────────

def sparse_to_sequences_preserve_order(X_sparse, shift=2):
    """
    Convert sparse matrix to sequences of feature indices while preserving word order.
    Each non-zero feature index appears in its original position in the document.
    """
    sequences = []
    for i in range(X_sparse.shape[0]):
        row = X_sparse[i].toarray().flatten()
        # Get all non-zero feature indices in their original order
        feature_indices = np.where(row > 0)[0]
        shifted_indices = (feature_indices + shift).tolist() if len(feature_indices) > 0 else [0]
        sequences.append(shifted_indices)
    return sequences

print("Converting to sequences...")
X_train_seqs = sparse_to_sequences_preserve_order(X_train_vec)
X_val_seqs = sparse_to_sequences_preserve_order(X_val_vec)
X_test_seqs = sparse_to_sequences_preserve_order(X_test_vec)

# ─────────────────────────────────────────
# VOCAB
# ─────────────────────────────────────────

vocab = {i: i for i in range(X_train_vec.shape[1] + 2)}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

print(f"Vocab size: {len(vocab)}")

# ─────────────────────────────────────────
# DATASET & DATALOADER
# ─────────────────────────────────────────

def collate_fn(batch):
    """Collate function to pad sequences in a batch."""
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    labels = torch.tensor(labels, dtype=torch.long)
    return padded, lengths, labels


class SequenceDataset(Dataset):
    def __init__(self, sequences, labels, vocab):
        self.labels = labels
        self.vocab = vocab
        self.encoded = [
            torch.tensor([vocab.get(t, vocab["<UNK>"]) for t in seq], dtype=torch.long)
            for seq in sequences
        ]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded[idx], self.labels[idx]


dataset_train = SequenceDataset(X_train_seqs, y_train_sub, vocab)
dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

dataset_val = SequenceDataset(X_val_seqs, y_val, vocab)
dataloader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

dataset_test = SequenceDataset(X_test_seqs, y_test, vocab)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print("Datasets ready")

# ─────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, 
                 dropout=0.3, rnn_type='lstm'):
        super().__init__()
        self.rnn_type = rnn_type.lower()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        rnn_class = nn.LSTM if self.rnn_type == 'lstm' else nn.GRU
        
        self.rnn = rnn_class(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, 2)

    def forward(self, x, lengths):
        emb = self.dropout(self.embedding(x))
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        
        if self.rnn_type == 'lstm':
            _, (hn, _) = self.rnn(packed)
        else:  # GRU
            _, hn = self.rnn(packed)
        
        hn = torch.cat([hn[-2], hn[-1]], dim=1)  # concat forward + backward
        return self.classifier(self.dropout(hn))


print(f"\nLoading model ({RNN_TYPE})...")
model = RNNClassifier(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    rnn_type=RNN_TYPE,
).to(device)

print(model)

# ─────────────────────────────────────────
# LOSS & OPTIMIZER
# ─────────────────────────────────────────

def compute_class_weights(labels):
    counts = torch.bincount(torch.tensor(labels)).float()
    weights = 1.0 / counts
    weights = weights / weights.sum()
    return weights

class_weights = compute_class_weights(y_train_sub).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

# ─────────────────────────────────────────
# METRICS HELPER
# ─────────────────────────────────────────

def compute_metrics(all_labels, all_preds, all_probs):
    """Compute evaluation metrics."""
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    
    if DATASET == "pres":
        return {
            "f1": float(f1_score(all_labels, all_preds, average="macro", zero_division=0)),
            "precision": float(precision_score(all_labels, all_preds, average="macro", zero_division=0)),
            "recall": float(recall_score(all_labels, all_preds, average="macro", zero_division=0)),
            "roc_auc": float(roc_auc_score(all_labels, all_probs)),
            "avg_precision": float(average_precision_score(all_labels, all_probs)),
        }
    
    return {
        "f1": float(f1_score(all_labels, all_preds, average="binary", zero_division=0)),
        "precision": float(precision_score(all_labels, all_preds, zero_division=0)),
        "recall": float(recall_score(all_labels, all_preds, zero_division=0)),
        "roc_auc": float(roc_auc_score(all_labels, all_probs)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
    }

# ─────────────────────────────────────────
# TRAIN / EVAL LOOP
# ─────────────────────────────────────────

def run_epoch(model, loader, optimizer, criterion, training=True):
    """Run one epoch of training or evaluation."""
    model.train() if training else model.eval()

    total_loss = 0
    all_labels, all_preds, all_probs = [], [], []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for padded, lengths, labels in loader:
            padded = padded.to(device)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad()

            logits = model(padded, lengths)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            total_loss += loss.item()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].detach().cpu().numpy())

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = float(total_loss / len(loader))
    return metrics

# ─────────────────────────────────────────
# TRAINING WITH EARLY STOPPING
# ─────────────────────────────────────────

print("\n" + "="*80)
print("TRAINING")
print("="*80)

history = []
best_f1 = 0.0
best_model_state = None
patience_counter = 0

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}", end=" ")

    train_m = run_epoch(model, dataloader_train, optimizer, criterion, training=True)
    val_m = run_epoch(model, dataloader_val, optimizer, criterion, training=False)

    # Early stopping
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        improvement_marker = "↑ (BEST)"
    else:
        patience_counter += 1
        improvement_marker = f"(patience {patience_counter}/{EARLY_STOPPING_PATIENCE})"

    history.append({
        "epoch": epoch + 1,
        "train": train_m,
        "val": val_m,
    })

    print(
        f"{improvement_marker:25s} | "
        f"loss  train {train_m['loss']:.4f}  val {val_m['loss']:.4f} | "
        f"f1    train {train_m['f1']:.3f}  val {val_m['f1']:.3f} | "
        f"auc   train {train_m['roc_auc']:.3f}  val {val_m['roc_auc']:.3f} | "
        f"prec  train {train_m['precision']:.3f}  val {val_m['precision']:.3f} | "
        f"rec   train {train_m['recall']:.3f}  val {val_m['recall']:.3f}"
    )

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}: No improvement for {EARLY_STOPPING_PATIENCE} epochs")
        break

print(f"\nBest val F1: {best_f1:.3f}")

# ─────────────────────────────────────────
# TEST EVALUATION
# ─────────────────────────────────────────

print("\n" + "="*80)
print("TEST EVALUATION")
print("="*80)

model.load_state_dict(best_model_state)
torch.save(model.state_dict(), f"{SAVING_FILE_NAME}.pth")

test_m = run_epoch(model, dataloader_test, optimizer, criterion, training=False)

print(
    f"\nTest | loss {test_m['loss']:.4f} | f1 {test_m['f1']:.4f} | "
    f"prec {test_m['precision']:.4f} | rec {test_m['recall']:.4f} | "
    f"auc {test_m['roc_auc']:.4f} | ap {test_m['avg_precision']:.4f}"
)

# ─────────────────────────────────────────
# SAVE RESULTS
# ─────────────────────────────────────────

results = {
    "run_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "device": str(device),
    "vectorizer": {
        "type": VECTORIZER_TYPE,
        "max_features": MAX_FEATURES,
        "ngram_range": NGRAM_RANGE,
    },
    "model_config": {
        "rnn_type": RNN_TYPE,
        "embed_dim": EMBED_DIM,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 1e-5,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    },
    "best_val_f1": best_f1,
    "epochs_trained": len(history),
    "history": history,
    "test": test_m,
}

with open(f"results_{SAVING_FILE_NAME}.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to results_{SAVING_FILE_NAME}.json")
print(f"Model saved to {SAVING_FILE_NAME}.pth")

print("DATASET ", DATASET)
print(f"Transformer embedding dimension: {EMBED_DIM}")
print(f"\nLoaded model ({RNN_TYPE})...")
